# Vector Search and Embeddings

## 1. Embeddings Setup and Sample Testing

Before performing vector search, we must convert text data into numerical representations called **embeddings**. These are dense vectors that capture the semantic meaning of sentences.

We use the **Sentence-Transformers (SBERT)** library, a popular open-source framework for state-of-the-art sentence, text, and image embeddings.

In [1]:
# Import the library to generate embeddings
from sentence_transformers import SentenceTransformer

# Load a pre-trained model. 'all-MiniLM-L6-v2' is efficient and high-performing.
# It maps sentences to a 384-dimensional dense vector space.
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Encoding a Single Sentence

Let's see how a raw string is transformed into a vector and inspect its dimensions.

In [2]:
# Sample question
q1 = "I just discovered the course, can I still join?"
# Encode text into a numerical vector
v1 = model.encode(q1)

# Check the dimensionality (expected 384 for this specific model)
v1.shape

(384,)

### Testing Semantic Similarity

When vectors represent similar meanings, they should have a high similarity score (close to 1).

In [3]:
# Compare three different sentences to see how semantic similarity varies

q1 = "Apple fruit is good for health"
q2 = "banana fruit is good for health"
q3 = "Apple share price hit 52 week max"

v1 = model.encode(q1)
v2 = model.encode(q2)
v3 = model.encode(q3)

# Higher values indicate higher similarity
print("q1 and q2 similartiy ", v1.dot(v2))
print("q1 and q3 similartiy ", v1.dot(v3))
print("q2 and q3 similartiy ", v2.dot(v3))


q1 and q2 similartiy  0.7682494
q1 and q3 similartiy  0.25341332
q2 and q3 similartiy  0.04609214


## 2. Distance and Angle

Because SBERT models usually produce normalized vectors, the **dot product** is equivalent to the **cosine similarity**. 

We can calculate the actual angle between vectors to visualize their closeness.

In [4]:
# Import numpy for mathematical operations
import numpy as np

### Scenario A: High Similarity
q1 = "I just discovered the LLM course, can I still join?"
v1 = model.encode(q1)

res = "You can still join the LLM course."
res_v1 = model.encode(res)

dv = v1.dot(res_v1) # Result ~0.91
print("dot value for q1 and res = ", dv)

# Convert similarity to radians then degrees
angle_rad = np.arccos(dv)
angle_deg = np.degrees(angle_rad)
print("cosine value for q1 and res = ", angle_deg)


### Scenario B: Low Similarity
res2 = "Kubernetes orchestatres several docker containers effectively."
res_v2 = model.encode(res2)
dv2 = v1.dot(res_v2)

# A large angle indicates the topics are unrelated
angle_rad = np.arccos(dv2)
angle_deg = np.degrees(angle_rad) # 97.945632
print("cosine of res2 = ", angle_deg)



dot value for q1 and res =  0.9101957
cosine value for q1 and res =  24.467587
cosine of res2 =  96.47518


## Embedding the FAQ dataset

Generate embeddings the FAQ dataset, 
- Load the documents to the memory
- Filter based on "question" and "answer" fields
- Created embeddings (convert into the vectors) based on the batch fashion
- Create vector search using input question (which is dot operation)
- Get top 5 results



In [5]:
# Fetch the ingestion helper script from the course repository
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py



7[Files: 0  Bytes: 0  [0 B/s] Re]87[https://raw.githubusercontent.]87Saving 'ingest.py.1'
87ingest.py.1          100% [=============================>]     340     --.-KB/s87HTTP response 200  [https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py]
87ingest.py.1          100% [=============================>]     340     --.-KB/s87[Files: 1  Bytes: 340  [1.30KB/]8

### Step 2: Load and Prepare Documents

In [6]:
from ingest import load_faq_data

# Extract documents from the JSON source
documents = load_faq_data()

print(f"Loaded {len(documents)} documents.")
documents[10]

Loaded 1342 documents.


{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

### Step 3: Text Consolidation

In [7]:
# Filter data by taking question and answer only
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

len(texts)

1342

In [8]:
# We use batches of 50 to avoid overloading memory and to provide progress updates
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1342

In [9]:
# Encode the user query
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

# Single comparison test
v1.dot(vectors[10])


np.float32(0.33153284)

In [10]:
# Manually iterate through the entire list to find similarities
# This is slow for very large datasets

scores = []

for i in range(len(vectors)):
    score = v1.dot(vectors[i])
    scores.append(score)

len(scores)
#scores[0]

1342

In [11]:
# Quick example of NumPy array shape properties

import numpy as np

arr = np.array([[1,11], [2,22], [3,33], [4,44]])
arr.shape

(4, 2)

In [12]:
import numpy as np
X = np.array(vectors)
X.shape   # X.shape returns (1342, 384) 

(1342, 384)

In [16]:
scores2 = X.dot(v1)
#scores2

In [14]:
import json

# Get index of the highest similarity score
idx = np.argmax(scores2)
print(idx, scores[idx])
print(q1, "\n\n", json.dumps(documents[idx], indent=2))

2 0.762941
Can I still join the course after the start date? 

 {
  "id": "3f1424af17",
  "course": "data-engineering-zoomcamp",
  "section": "General Course-Related Questions",
  "question": "Course: Can I still join the course after the start date?",
  "answer": "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."
}


## 5. Ranking Top Results

Usually, we want more than one result. We use `argsort` to find the indices of the top-k matches.

In [15]:
# argsort sorts from lowest to highest; we take the last 5 and reverse the list
top5 = np.argsort(scores2)[-5:]
top5 = top5[::-1]

print("Top 5 indices = ", top5)

# Display the top matches and their respective similarity scores
for idx in top5:
    print(scores2[idx])
    print(documents[idx])
    print()


Top 5 indices =  [  2 617 899 536   7]
0.762941
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.7579371
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.71921325
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomca